# Macroinvertebrates Database Schema

In [1]:
import pandas as pd
import requests
import json
import queryUSGS as q

In [2]:
# Config cell
DB_PATH = "macroinvertebrates.db"

In [3]:
# SET UP 
import sqlite3

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")

def run_sql(sql: str):
    try:
        cur = conn.execute(sql)
        if sql.lstrip().upper().startswith(("SELECT", "PRAGMA")):
            rows = cur.fetchall()
            print(rows)
        else:
            conn.commit()
            print("OK")
    except sqlite3.Error as err:
        print(f"SQLite error: {err}")

In [8]:
def create_tables(conn):
    
    cur = conn.cursor()
    cur.executescript("""
    DROP TABLE IF EXISTS Year;
    DROP TABLE IF EXISTS Date;
    DROP TABLE IF EXISTS Sample;
    DROP TABLE IF EXISTS Macroinvertebrate;
    DROP TABLE IF EXISTS Taxonomy;
    """)


    cur.executescript("""

    CREATE TABLE Year (
        year INT CHECK(year >= 2019),
        winterSediment REAL,
        PRIMARY KEY(year)
    );
        
    CREATE TABLE Date (
        date VARCHAR(5),   
        startTime TEXT,
        year INT,
        monthlyAveragePrecipitation REAL,
        maxDischarge REAL,
        medianDischarge REAL,
        aggregatedDegreeDays REAL,
        maxTurb REAL,
        medianTurb REAL,
        maxWaterTemp REAL,
        medWaterTemp REAL,
        season TEXT CHECK(season=='Spring' OR season=='Fall' OR season=='Summer'),
        PRIMARY KEY(date, startTime),
        FOREIGN KEY(startTime) REFERENCES Sample(startTime),
        FOREIGN KEY(year) REFERENCES Year(year)
    );

        
    CREATE TABLE Sample (
        sampleID VARCHAR(12),  
        date VARCHAR(5), 
        year INT,
        startTime TEXT, 
        sampleMethod TEXT,
        quadrat REAL,
        location TEXT CHECK(location=='Upstream' OR location =='Downstream'),
        microhabitat TEXT CHECK(microhabitat == 'DSR' OR microhabitat == 'DSP' OR microhabitat == 'DFR' OR microhabitat == 'DM' OR microhabitat ==  'DSH' OR microhabitat == 'USR' OR microhabitat == 'UFR' OR microhabitat == 'UM' OR  microhabitat == 'USH' OR microhabitat == 'USU' OR microhabitat == NULL),
        waterDepth REAL,
        percentSediment REAL,
        percentRock REAL CHECK (percentRock >= 0 AND percentRock <= 100),
        percentOrganic REAL CHECK (percentOrganic>=0 AND percentOrganic <= 100),
        pH REAL CHECK (pH >= 0 AND pH <= 14),
        waterTemp REAL,
        dissolvedO2 REAL,
        light REAL,
        flow REAL,
        turb REAL, 
        conductivity REAL, 
        nitrate REAL,
        alkalinity INT,
        PRIMARY KEY(sampleID, startTime, quadrat),
        FOREIGN KEY(date) REFERENCES Date(date)
    );

    CREATE TABLE Macroinvertebrate (
        scientificName TEXT,  
        sampleID VARCHAR(12),
        quadrat REAL,
        stage REAL CHECK(stage == 'larva' OR stage == 'pupa' OR stage == 'adult'),
        numOrganismsFound INT CHECK (numOrganismsFound >= 0),
        benthicArea REAL,
        invertebrateDensity REAL,
        PRIMARY KEY(scientificName, sampleID, quadrat),
        FOREIGN KEY(sampleID) REFERENCES Sample(sampleID)
        FOREIGN KEY(quadrat) REFERENCES Sample(quadrat)
    );

    CREATE TABLE Taxonomy (
        taxonID VARCHAR(6) UNIQUE,      
        scientificName TEXT UNIQUE,
        taxonRank TEXT,
        taxgroup TEXT,
        phylum TEXT,
        subphylum TEXT,
        class TEXT,
        taxOrder TEXT,
        family TEXT,
        subfamily TEXT,
        tribe TEXT,
        genus TEXT,
        taxaNotes TEXT, 
        tolerance INT,
        ffg VARCHAR(3) CHECK(ffg == 'cf' OR ffg == 'cg' or ffg =='om' OR ffg == 'prc' OR ffg == 'prd' OR ffg == 'scr' OR ffg == 'sh'), 
        ffg2 VARCHAR(3) CHECK(ffg2 == 'cf' OR ffg2 == 'cg' or ffg2 =='om' OR ffg2 == 'prc' OR ffg2 == 'prd' OR ffg2 == 'scr' OR ffg2 == 'sh' OR ffg2 == NULL),
        PRIMARY KEY(taxonID),
        FOREIGN KEY(scientificName) REFERENCES Macroinvertebrate(scientificName)
    );
 """)

    conn.commit()

def main():
    conn = sqlite3.connect(DB_PATH)
    create_tables(conn)
    cur = conn.cursor()

    add_data(conn, cur)

    conn.commit()
    
if __name__ == "__main__":
    main();

Reading in .csv files...


In [3]:
''' 
Searches for files by the names of env.csv, macros.csv, master.taxa.csv, and waterQ.csv. Inserts data into database. 

DOES NOT CHECK IF ENTRY ALREADY EXISTS
'''

def add_data(con, cur):

    print('Reading in .csv files...')

    try:
        env_df = pd.read_csv("env.csv")
    except:
        print("env.csv not found") 

    try:
        macro_df = pd.read_csv("macros.csv")
    except:
        print("env.csv not found") 

    try:
        taxa_df = pd.read_csv("master.taxa.csv")
    except:
        print("env.csv not found") 

    try:
        water_quality_df = pd.read_csv("waterQ.csv")
    except:
        print("waterQ.csv not found") 
    
    env_df = pd.DataFrame(env_df)
    env_df.rename(columns = {
        'mon.precip' : 'monthlyAveragePrecipitation',
        'mon.ADD' : 'aggregatedDegreeDays',
        'mon.max.discharge' : 'maxDischarge',
        'mon.median.discharge' : 'medianDischarge',
        'mon.max.turb' : 'maxTurb',
        'mon.median.turb' : 'medianTurb',
        'mon.max.wTemp' : 'maxWaterTemp',
        'mon.median.wTemp' : 'medWaterTemp',
        'per_sediment' : 'percentSediment',
        'winter.sediment' : 'winterSediment',
        'per_rock' : 'percentRock',
        'per_organic' : 'percentOrganic',
        'wTemp': 'waterTemp',
        'DO' : 'dissolvedO2',
        'cond' : 'conductivity',
        'depth' : 'waterDepth'
    }, inplace = True)

    macro_df = pd.DataFrame(macro_df)
    macro_df.rename(columns = {
        'number' :'numOrganismsFound',
        'invDens' : 'invertebrateDensity'
    }, inplace = True)


    taxa_df = pd.DataFrame(taxa_df)
    taxa_df.rename(columns = {
        'taxon_group' :'taxgroup',
        'acceptedTaxonID' : 'taxonID',
        'taxa.notes' : 'taxaNotes',
        'order' : 'taxOrder',
        'FFG' : 'ffg',
        'FFG2' : 'ffg2'
    }, inplace = True)
    

    water_quality_df = pd.DataFrame(water_quality_df)
    water_quality_df.rename(columns = {
        'depth' : 'waterDepth',
        'wTemp' : 'waterTemp',
        'DO': 'dissolvedO2',
        'cond': 'conductivity'
    }, inplace = True)

    year_subtable = env_df[['year','winterSediment']].sort_values(by='winterSediment').drop_duplicates().loc[(env_df != 0).all(1)]

    taxa_subtable = taxa_df[['taxonID', 'scientificName', 'taxonRank','taxgroup','phylum', 'subphylum','class','taxOrder', 'family','subfamily','tribe', 'genus','taxaNotes', 'tolerance', 'ffg', 'ffg2']]

    date_subtable = env_df[['date','year','monthlyAveragePrecipitation','maxDischarge','medianDischarge','aggregatedDegreeDays','maxTurb','medianTurb','maxWaterTemp', 'medWaterTemp','season']].drop_duplicates()

    merged_df = water_quality_df.join(env_df.set_index('sampleID'), on='sampleID', how='outer', lsuffix='_wq', rsuffix='_env')
    merged_df.reset_index(inplace = True)
    
    sample_subtable = merged_df[['sampleID', 'year_wq', 'startTime', 'location_wq','microhabitat_wq','waterDepth_wq', 'percentSediment','percentRock','percentOrganic','pH_wq','waterTemp_wq','dissolvedO2_wq', 'light_wq', 'flow_wq', 'turb_wq','conductivity_wq','nitrate_wq','alkalinity_wq', 'quadrat']]

    macros_subtable = macro_df[['scientificName', 'sampleID', 'stage', 'numOrganismsFound', 'benthicArea','invertebrateDensity']]


    try:
        cur.executemany(
                'INSERT INTO "Macroinvertebrate" ("scientificName", "sampleID","stage", "numOrganismsFound","benthicArea","invertebrateDensity") VALUES (?, ?, ?, ?, ?, ?);', macros_subtable.values.tolist()
        )
    except:
        print('Issue inserting into Macroinvertebrate table.')

    try:
        cur.executemany(
                'INSERT INTO "Taxonomy" ("taxonID", "scientificName", "taxonRank","taxgroup","phylum", "subphylum","class","taxOrder", "family","subfamily","tribe", "genus","taxaNotes", "tolerance", "ffg", "ffg2") VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);', taxa_subtable.values.tolist()
        )
    except:
        print('Issue inserting into Taxonomy table.')

    try:
        cur.executemany(
                'INSERT INTO "Sample" ("sampleID", "startTime", "year", "location","microhabitat","waterDepth", "percentSediment","percentRock","percentOrganic","pH","waterTemp","dissolvedO2", "light", "flow", "turb","conductivity","nitrate","alkalinity", "quadrat") VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);', sample_subtable.values.tolist()
        )
    except:
        print('Issue inserting into Sample table.')

    try:
        cur.executemany(
               'INSERT INTO "Date" ("date","year","monthlyAveragePrecipitation","maxDischarge","medianDischarge","aggregatedDegreeDays","maxTurb","medianTurb","maxWaterTemp", "medWaterTemp","season") VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);', date_subtable.values.tolist()
        )
    except:
        print('Issue inserting into Date table.')

    try:
        cur.executemany(
            'INSERT INTO "Year" ("year", "winterSediment") VALUES (?, ?);', year_subtable.values.tolist()
        )
    except:
         print('Issue inserting into Year table.')

    conn.commit()

In [2]:
try:
    env_df = pd.read_csv("env.csv")
except:
    print("env.csv not found") 

try:
    macro_df = pd.read_csv("macros.csv")
except:
    print("env.csv not found") 

try:
    taxa_df = pd.read_csv("master.taxa.csv")
except:
    print("env.csv not found") 

try:
    water_quality_df = pd.read_csv("waterQ.csv")
except:
    print("waterQ.csv not found") 

env_df = pd.DataFrame(env_df)
env_df.rename(columns = {
    'mon.precip' : 'monthlyAveragePrecipitation',
    'mon.ADD' : 'aggregatedDegreeDays',
    'mon.max.discharge' : 'maxDischarge',
    'mon.median.discharge' : 'medianDischarge',
    'mon.max.turb' : 'maxTurb',
    'mon.median.turb' : 'medianTurb',
    'mon.max.wTemp' : 'maxWaterTemp',
    'mon.median.wTemp' : 'medWaterTemp',
    'per_sediment' : 'percentSediment',
    'winter.sediment' : 'winterSediment',
    'per_rock' : 'percentRock',
    'per_organic' : 'percentOrganic',
    'wTemp': 'waterTemp',
    'DO' : 'dissolvedO2',
    'cond' : 'conductivity',
    'depth' : 'waterDepth'
}, inplace = True)

macro_df = pd.DataFrame(macro_df)
macro_df.rename(columns = {
    'number' :'numOrganismsFound',
    'invDens' : 'invertebrateDensity'
}, inplace = True)


taxa_df = pd.DataFrame(taxa_df)
taxa_df.rename(columns = {
    'taxon_group' :'taxgroup',
    'acceptedTaxonID' : 'taxonID',
    'taxa.notes' : 'taxaNotes',
    'order' : 'taxOrder',
    'FFG' : 'ffg',
    'FFG2' : 'ffg2'
}, inplace = True)


water_quality_df = pd.DataFrame(water_quality_df)
water_quality_df.rename(columns = {
    'depth' : 'waterDepth',
    'wTemp' : 'waterTemp',
    'DO': 'dissolvedO2',
    'cond': 'conductivity'
}, inplace = True)
    
year_subtable = env_df[['year','winterSediment']].sort_values(by='winterSediment').drop_duplicates().loc[(env_df != 0).all(1)]

taxa_subtable = taxa_df[['taxonID', 'scientificName', 'taxonRank','taxgroup','phylum', 'subphylum','class','taxOrder', 'family','subfamily','tribe', 'genus','taxaNotes', 'tolerance', 'ffg', 'ffg2']]

date_subtable = env_df[['date','year','monthlyAveragePrecipitation','maxDischarge','medianDischarge','aggregatedDegreeDays','maxTurb','medianTurb','maxWaterTemp', 'medWaterTemp','season']].drop_duplicates()

# Add discharge and gageHeight to date
discharge = []

for i, row in date_subtable.iterrows():
    date = row['date']
    value = q.get_discharge(date)
    discharge.append(value)

date_subtable['discharge'] = discharge


gageHeight = []
for i, row in date_subtable.iterrows():
    date = row['date']
    value = q.get_gage_Height(date)
    gageHeight.append(value)


date_subtable['gageHeight'] = gageHeight

merged_df = water_quality_df.join(env_df.set_index('sampleID'), on='sampleID', how='outer', lsuffix='_wq', rsuffix='_env')
merged_df.reset_index(inplace = True)

sample_subtable = merged_df[['sampleID', 'year_wq', 'startTime', 'location_wq','microhabitat_wq','waterDepth_wq', 'percentSediment','percentRock','percentOrganic','pH_wq','waterTemp_wq','dissolvedO2_wq', 'light_wq', 'flow_wq', 'turb_wq','conductivity_wq','nitrate_wq','alkalinity_wq', 'quadrat']]

macros_subtable = macro_df[['scientificName', 'sampleID', 'stage', 'numOrganismsFound', 'benthicArea','invertebrateDensity']]





<bound method NDFrame.head of            date  year  monthlyAveragePrecipitation  maxDischarge  \
0    2019-05-28  2019                         69.8         554.0   
5    2019-05-30  2019                         75.9         324.0   
10   2019-05-31  2019                         78.7         324.0   
15   2019-06-03  2019                         65.2         321.0   
20   2019-06-04  2019                         62.4         313.0   
..          ...   ...                          ...           ...   
310  2024-06-12  2024                         76.2         481.0   
315  2024-09-18  2024                         28.2         202.0   
320  2024-09-19  2024                         13.5         202.0   
325  2024-09-23  2024                         13.0          40.1   
330  2024-09-25  2024                         12.4          32.8   

     medianDischarge  aggregatedDegreeDays  maxTurb  medianTurb  maxWaterTemp  \
0              137.0                384.75   1557.0      1.0000         